# Sales Pipeline - EDA and Data Processing

This notebook contains the main sales pipeline for data exploration, transformation, and analysis.

This notebook builds a simple medallion-style pipeline (bronze → silver → gold) using Polars + DuckDB. In production this would correspond to PySpark tables in Databricks with Unity Catalog, orchestrated by Prefect.

In [64]:
# Dependency check
try:
    import polars as pl
    import duckdb
    from pathlib import Path

    print(f"✅ Polars version: {pl.__version__}")
    print(f"✅ DuckDB version: {duckdb.__version__}")
    print("\n🎉 All dependencies are ready!")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("\nPlease install packages with: python3 -m pip install polars duckdb")

✅ Polars version: 1.35.2
✅ DuckDB version: 1.4.2

🎉 All dependencies are ready!


In [65]:
from pathlib import Path
import polars as pl
import duckdb

# Find project root by navigating up from current directory until we find the 'data' folder
# This works regardless of where the notebook is run from
current = Path.cwd()
while current != current.parent:
    if (current / "data").exists() and (current / "data" / "countries.json").exists():
        ROOT_DIR = current
        break
    current = current.parent
else:
    # Fallback: assume we're in src/EDA and go up 2 levels
    ROOT_DIR = Path.cwd().parent.parent

DATA_DIR = ROOT_DIR / "data"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Make sure layer folders exist under the REAL data directory
for d in (BRONZE_DIR, SILVER_DIR, GOLD_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT_DIR}")
print(f"Data directory: {DATA_DIR}")
ROOT_DIR, DATA_DIR, BRONZE_DIR, SILVER_DIR, GOLD_DIR

Project root: /Users/dr.jenniferemberton/Documents/GitHub/sales_test
Data directory: /Users/dr.jenniferemberton/Documents/GitHub/sales_test/data


(PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test'),
 PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data'),
 PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze'),
 PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver'),
 PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/gold'))

# Loader Helper

In [66]:
import json
from pathlib import Path
import polars as pl

def load_multi_json(path: Path) -> pl.DataFrame:
    """
    Load a file that contains many JSON objects concatenated together
    (e.g., {...}{...}{...}) into a Polars DataFrame.
    Works even if they are separated by commas/newlines/spaces.
    """
    text = path.read_text()
    records = []

    buf = []
    depth = 0
    in_obj = False

    for ch in text:
        if ch == "{":
            in_obj = True
            depth += 1
        if in_obj:
            buf.append(ch)
        if ch == "}":
            depth -= 1
            if depth == 0 and in_obj:
                obj_str = "".join(buf)
                records.append(json.loads(obj_str))
                buf = []
                in_obj = False

    return pl.DataFrame(records)


### Bronze Layer – Load Raw Data

In [67]:
# ---- Bronze layer: load raw JSON data ----

countries_raw = load_multi_json(DATA_DIR / "countries.json")
customers_raw = load_multi_json(DATA_DIR / "customers.json")
orders_raw    = load_multi_json(DATA_DIR / "orders.json")
products_raw  = load_multi_json(DATA_DIR / "products.json")
sales_raw     = load_multi_json(DATA_DIR / "sales.json")

# Peek at schemas / column names for later joins
for name, df in [
    ("countries", countries_raw),
    ("customers", customers_raw),
    ("orders", orders_raw),
    ("products", products_raw),
    ("sales", sales_raw),
]:
    print(f"\n{name} columns:", df.columns)
    display(df.head())


countries columns: ['Country', 'Currency', 'Name', 'Region', 'Population', 'Area (sq. mi.)', 'Pop. Density (per sq. mi.)', 'Coastline (coast per area ratio)', 'Net migration', 'Infant mortality (per 1000 births)', 'GDP ($ per capita)', 'Literacy (%)', 'Phones (per 1000)', 'Arable (%)', 'Crops (%)', 'Other (%)', 'Climate', 'Birthrate', 'Deathrate', 'Agriculture', 'Industry', 'Service']


Country,Currency,Name,Region,Population,Area (sq. mi.),Pop. Density (per sq. mi.),Coastline (coast per area ratio),Net migration,Infant mortality (per 1000 births),GDP ($ per capita),Literacy (%),Phones (per 1000),Arable (%),Crops (%),Other (%),Climate,Birthrate,Deathrate,Agriculture,Industry,Service
str,str,str,str,i64,i64,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""AD""","""EUR""","""Andorra""","""WESTERN EUROPE""",71201,468,152.1,0.0,6.6,4.05,19000,100.0,497.2,2.22,0.0,97.78,3.0,8.71,6.25,null,null,null
"""AE""","""AED""","""United Arab Emirates""","""NEAR EAST""",2602713,82880,31.4,1.59,1.03,14.51,23200,77.9,475.3,0.6,2.25,97.15,1.0,18.96,4.4,0.04,0.585,0.375
"""AF""","""AFN""","""Afghanistan""","""ASIA (EX. NEAR EAST)""",31056997,647500,48.0,0.0,23.06,163.07,700,36.0,3.2,12.13,0.22,87.65,1.0,46.6,20.34,0.38,0.24,0.38
"""AG""","""XCD""","""Antigua & Barbuda""","""LATIN AMER. & CARIB""",69108,443,156.0,34.54,-6.15,19.46,11000,89.0,549.9,18.18,4.55,77.27,2.0,16.93,5.37,0.038,0.22,0.743
"""AI""","""XCD""","""Anguilla""","""LATIN AMER. & CARIB""",13477,102,132.1,59.8,10.76,21.03,8600,95.0,460.0,0.0,0.0,100.0,2.0,14.17,5.34,0.04,0.18,0.78



customers columns: ['CustomerId', 'Active', 'Name', 'Address', 'City', 'Country', 'Email']


CustomerId,Active,Name,Address,City,Country,Email
i64,bool,str,str,str,str,str
1,true,"""Jason Orr""","""Ap #387-8229 Nullam Road""","""Kilsyth""","""HN""","""lacus.Nulla@Classaptenttaciti.…"
2,true,"""Devin Herman""","""P.O. Box 905, 9608 Etiam St.""","""Portici""","""HR""","""Integer.vulputate.risus@est.co…"
3,true,"""Kennan Head""","""Ap #694-3226 Odio St.""","""Bothey""","""JO""","""ligula@porttitortellusnon.org"""
4,false,"""Peter Fry""","""Ap #378-2594 Arcu. Road""","""Oudegem""","""LR""","""eu@Phasellusvitaemauris.com"""
5,true,"""Guy Ball""","""418-8277 Sociis Ave""","""Charleville-Mézières""","""TG""","""vestibulum.Mauris.magna@ipsumd…"



orders columns: ['OrderId', 'CustomerId', 'Date']


OrderId,CustomerId,Date
i64,i64,str
1,181,"""2018-01-01"""
2,119,"""2018-01-01"""
3,69,"""2018-01-01"""
4,173,"""2018-01-01"""
5,237,"""2018-01-01"""



products columns: ['ProductId', 'Name', 'ManufacturedCountry', 'WeightGrams']


ProductId,Name,ManufacturedCountry,WeightGrams
i64,str,str,i64
1,"""Generac""","""GB""",55
2,"""Ambigue""","""US""",128
3,"""Vaguee""","""CA""",103
4,"""Dubioum""","""PR""",77
5,"""Fuzzi""","""SG""",83



sales columns: ['SaleId', 'OrderId', 'ProductId', 'Quantity']


SaleId,OrderId,ProductId,Quantity
i64,i64,i64,i64
1,1,3,1
2,1,7,9
3,2,5,12
4,2,10,3
5,2,4,3


#### Persist bronze layer to Parquet

In [68]:
# Persist raw data into bronze layer as Parquet

countries_raw.write_parquet(BRONZE_DIR / "countries.parquet")
customers_raw.write_parquet(BRONZE_DIR / "customers.parquet")
orders_raw.write_parquet(BRONZE_DIR / "orders.parquet")
products_raw.write_parquet(BRONZE_DIR / "products.parquet")
sales_raw.write_parquet(BRONZE_DIR / "sales.parquet")

BRONZE_DIR, list(BRONZE_DIR.glob("*.parquet"))


(PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze'),
 [PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze/products.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze/orders.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze/sales.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze/customers.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/bronze/countries.parquet')])

## Silver Layer – Dimensional Model (Customers, Products, Countries, Fact Sales)

In [69]:
# Silver layer: read from bronze Parquet tables

countries_bronze = pl.read_parquet(BRONZE_DIR / "countries.parquet")
customers_bronze = pl.read_parquet(BRONZE_DIR / "customers.parquet")
orders_bronze    = pl.read_parquet(BRONZE_DIR / "orders.parquet")
products_bronze  = pl.read_parquet(BRONZE_DIR / "products.parquet")
sales_bronze     = pl.read_parquet(BRONZE_DIR / "sales.parquet")

countries_bronze.head(), customers_bronze.head(), orders_bronze.head()

(shape: (5, 22)
 ┌─────────┬──────────┬─────────────┬─────────────┬───┬───────────┬────────────┬──────────┬─────────┐
 │ Country ┆ Currency ┆ Name        ┆ Region      ┆ … ┆ Deathrate ┆ Agricultur ┆ Industry ┆ Service │
 │ ---     ┆ ---      ┆ ---         ┆ ---         ┆   ┆ ---       ┆ e          ┆ ---      ┆ ---     │
 │ str     ┆ str      ┆ str         ┆ str         ┆   ┆ f64       ┆ ---        ┆ f64      ┆ f64     │
 │         ┆          ┆             ┆             ┆   ┆           ┆ f64        ┆          ┆         │
 ╞═════════╪══════════╪═════════════╪═════════════╪═══╪═══════════╪════════════╪══════════╪═════════╡
 │ AD      ┆ EUR      ┆ Andorra     ┆ WESTERN     ┆ … ┆ 6.25      ┆ null       ┆ null     ┆ null    │
 │         ┆          ┆             ┆ EUROPE      ┆   ┆           ┆            ┆          ┆         │
 │ AE      ┆ AED      ┆ United Arab ┆ NEAR EAST   ┆ … ┆ 4.4       ┆ 0.04       ┆ 0.585    ┆ 0.375   │
 │         ┆          ┆ Emirates    ┆             ┆   ┆           

In [70]:
# -------------------------
# Silver dimensions
# -------------------------

# Country dimension: keep business-friendly geo info
dim_country = (
    countries_bronze
    .select(
        "Country",           # country code
        "Name",              # country name
        "Region",
        "Population",
        "GDP ($ per capita)",
    )
    .rename({
        "Country": "CountryCode",
        "Name": "CountryName",
        "GDP ($ per capita)": "GdpPerCapita",
    })
    .sort("CountryCode")
)

# Customer dimension: rename fields to be more descriptive
dim_customer = (
    customers_bronze
    .select(
        "CustomerId",
        "Active",
        "Name",
        "Address",
        "City",
        "Country",   
        "Email",
    )
    .rename({
        "Name": "CustomerName",
        "Country": "CountryCode",
    })
    .sort("CustomerId")
)

# Product dimension
dim_product = (
    products_bronze
    .select(
        "ProductId",
        "Name",
        "ManufacturedCountry",
        "WeightGrams",
    )
    .rename({
        "Name": "ProductName",
    })
    .sort("ProductId")
)

print("dim_country:", dim_country.shape)
print("dim_customer:", dim_customer.shape)
print("dim_product:", dim_product.shape)

dim_country.head(), dim_customer.head(), dim_product.head()

dim_country: (224, 5)
dim_customer: (400, 7)
dim_product: (12, 4)


(shape: (5, 5)
 ┌─────────────┬──────────────────────┬──────────────────────┬────────────┬──────────────┐
 │ CountryCode ┆ CountryName          ┆ Region               ┆ Population ┆ GdpPerCapita │
 │ ---         ┆ ---                  ┆ ---                  ┆ ---        ┆ ---          │
 │ str         ┆ str                  ┆ str                  ┆ i64        ┆ i64          │
 ╞═════════════╪══════════════════════╪══════════════════════╪════════════╪══════════════╡
 │ AD          ┆ Andorra              ┆ WESTERN EUROPE       ┆ 71201      ┆ 19000        │
 │ AE          ┆ United Arab Emirates ┆ NEAR EAST            ┆ 2602713    ┆ 23200        │
 │ AF          ┆ Afghanistan          ┆ ASIA (EX. NEAR EAST) ┆ 31056997   ┆ 700          │
 │ AG          ┆ Antigua & Barbuda    ┆ LATIN AMER. & CARIB  ┆ 69108      ┆ 11000        │
 │ AI          ┆ Anguilla             ┆ LATIN AMER. & CARIB  ┆ 13477      ┆ 8600         │
 └─────────────┴──────────────────────┴──────────────────────┴────────────┴

In [71]:
# -------------------------
# Clean orders: add OrderDate
# -------------------------
orders_clean = (
    orders_bronze
    .with_columns(
        pl.col("Date")
          .str.strptime(pl.Date, "%Y-%m-%d") 
          .alias("OrderDate")
    )
    .drop("Date")
)

# -------------------------
# Silver fact table: fact_sales_enriched
# -------------------------

fact_sales_enriched = (
    sales_bronze
    # Add order info (OrderDate, CustomerId)
    .join(orders_clean, on="OrderId", how="left")
    # Add customer info (uses CountryCode from dim_customer)
    .join(
        dim_customer.select(
            "CustomerId",
            "CustomerName",
            "City",
            "CountryCode",
            "Active",
        ),
        on="CustomerId",
        how="left",
    )
    # Add product info
    .join(
        dim_product.select(
            "ProductId",
            "ProductName",
            "ManufacturedCountry",
            "WeightGrams",
        ),
        on="ProductId",
        how="left",
    )
    # Add country / region info via CountryCode
    .join(
        dim_country.select(
            "CountryCode",
            "CountryName",
            "Region",
            "GdpPerCapita",
        ),
        on="CountryCode",
        how="left",
    )
    .select(
        "SaleId",
        "OrderId",
        "OrderDate",
        "CustomerId",
        "CustomerName",
        "City",
        "CountryCode",
        "CountryName",
        "Region",
        "GdpPerCapita",
        "ProductId",
        "ProductName",
        "ManufacturedCountry",
        "WeightGrams",
        "Quantity",
        "Active",          # is the customer active?
    )
    .sort(["OrderDate", "SaleId"])
)

print("fact_sales_enriched:", fact_sales_enriched.shape)
fact_sales_enriched.head()



fact_sales_enriched: (61883, 16)


SaleId,OrderId,OrderDate,CustomerId,CustomerName,City,CountryCode,CountryName,Region,GdpPerCapita,ProductId,ProductName,ManufacturedCountry,WeightGrams,Quantity,Active
i64,i64,date,i64,str,str,str,str,str,i64,i64,str,str,i64,i64,bool
1,1,2018-01-01,181,"""Peter Diaz""","""Camrose""","""MX""","""Mexico""","""LATIN AMER. & CARIB""",9000,3,"""Vaguee""","""CA""",103,1,true
2,1,2018-01-01,181,"""Peter Diaz""","""Camrose""","""MX""","""Mexico""","""LATIN AMER. & CARIB""",9000,7,"""Nebuloon""","""CH""",118,9,true
3,2,2018-01-01,119,"""Stewart Cain""","""Maria""","""AZ""","""Azerbaijan""","""C.W. OF IND. STATES""",3400,5,"""Fuzzi""","""SG""",83,12,true
4,2,2018-01-01,119,"""Stewart Cain""","""Maria""","""AZ""","""Azerbaijan""","""C.W. OF IND. STATES""",3400,10,"""Uncleary""","""PE""",93,3,true
5,2,2018-01-01,119,"""Stewart Cain""","""Maria""","""AZ""","""Azerbaijan""","""C.W. OF IND. STATES""",3400,4,"""Dubioum""","""PR""",77,3,true


In [72]:
# -------------------------
# Persist silver layer to Parquet
# -------------------------

dim_country.write_parquet(SILVER_DIR / "dim_country.parquet")
dim_customer.write_parquet(SILVER_DIR / "dim_customer.parquet")
dim_product.write_parquet(SILVER_DIR / "dim_product.parquet")
fact_sales_enriched.write_parquet(SILVER_DIR / "fact_sales_enriched.parquet")

SILVER_DIR, list(SILVER_DIR.glob("*.parquet"))

(PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver'),
 [PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver/fact_sales_enriched.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver/dim_country.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver/dim_product.parquet'),
  PosixPath('/Users/dr.jenniferemberton/Documents/GitHub/sales_test/data/silver/dim_customer.parquet')])